# Paper Summary Tables

The selected judges are GPT-5.4 mini, Qwen 3.6 35B, and Gemma 4 31B. Llama is excluded. Failed or missing ratings are excluded from means, sample SDs, and agreement, and valid judge counts are reported explicitly.

LLM metrics cover all three question types. Final and total recall cover only entity questions, excluding empty annotations. Overall means pool valid item-level scores within each judge before averaging judge means with equal weight.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

REPO_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "evaluation_results.py").exists())
BASE_DIR = REPO_ROOT / "evaluations" / "biomni-base"
ANSWER_REPORT = "original"
WINRATE_DIRNAME = f"answer_winrate-{ANSWER_REPORT}"
JUDGE_IDS = ["gpt-5.4-mini", "qwen3.6-35b-a3b", "gemma-4-31b"]
JUDGE_LABELS = {
    "gemma-4-31b": "Gemma 4 31B",
    "gpt-5.4-mini": "GPT-5.4 mini",
    "qwen3.6-35b-a3b": "Qwen 3.6 35B",
}

TABLE_OUTPUT_DIR = BASE_DIR / "paper_tables" / "judges" / "ensemble"
TABLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def read_csv_or_empty(path):
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()


def judge_analysis_dir(judge_id):
    return BASE_DIR / "analysis" / "judges" / judge_id

BASE_DIR

PosixPath('/home/desild/work/research/LLM-Workflow-Explorer/evaluations/biomni-base')

## Load and Aggregate Judge Results

Each `*_mean` value is the equal-weight mean of the available judge-level means. Each corresponding `*_std` is the sample standard deviation across those judge-level means, not the within-question standard deviation stored in an individual judge's source summary. Per-judge source rows are retained in `judge_sources` and exported below.

In [2]:
LLM_METRICS = ["llm_completeness","llm_relevance","llm_faithfulness","llm_understanderbility"]
CATEGORY_NAMES = ["bool", "entity", "numeric"]
availability_rows = []
per_judge_summaries = {category: [] for category in CATEGORY_NAMES}
winrate_frames = []
pairwise_frames = []

for judge_id in JUDGE_IDS:
    analysis_dir = judge_analysis_dir(judge_id)
    for category in CATEGORY_NAMES:
        path = analysis_dir / category / "results_summary.csv"
        frame = read_csv_or_empty(path)
        raw = read_csv_or_empty(analysis_dir / category / "results.csv")
        llm_values = raw.reindex(columns=LLM_METRICS).apply(pd.to_numeric, errors="coerce")
        availability_rows.append({
            "judge_id": judge_id,
            "judge": JUDGE_LABELS[judge_id],
            "artifact": f"{category}_summary",
            "available": not frame.empty,
            "valid_llm_rows": int(llm_values.notna().all(axis=1).sum()),
            "llm_error_rows": int(raw.get("llm_answer_quality_error", pd.Series(dtype=str)).notna().sum()),
            "path": str(path.relative_to(REPO_ROOT)),
        })
        if not frame.empty:
            if category != "entity":
                frame = frame.drop(columns=[column for column in frame if column.startswith("entity_")])
            elif not raw.empty and "entity_recall_final" in raw:
                frame["entity_recall_final_count"] = frame["run"].map(
                    raw.groupby("run")["entity_recall_final"].count()
                )
            frame.insert(0, "judge_id", judge_id)
            frame.insert(1, "judge", JUDGE_LABELS[judge_id])
            per_judge_summaries[category].append(frame)

    winrate_path = analysis_dir / WINRATE_DIRNAME / "answer_winrate_summary.csv"
    winrate = read_csv_or_empty(winrate_path)
    availability_rows.append({
        "judge_id": judge_id,
        "judge": JUDGE_LABELS[judge_id],
        "artifact": "answer_winrate_summary",
        "available": not winrate.empty,
        "path": str(winrate_path.relative_to(REPO_ROOT)),
    })
    if not winrate.empty:
        winrate.insert(0, "judge_id", judge_id)
        winrate.insert(1, "judge", JUDGE_LABELS[judge_id])
        winrate_frames.append(winrate)

    pairwise_path = analysis_dir / WINRATE_DIRNAME / "pairwise_answer_winrate.csv"
    pairwise = read_csv_or_empty(pairwise_path)
    availability_rows.append({
        "judge_id": judge_id,
        "judge": JUDGE_LABELS[judge_id],
        "artifact": "pairwise_answer_winrate",
        "available": not pairwise.empty,
        "path": str(pairwise_path.relative_to(REPO_ROOT)),
    })
    if not pairwise.empty:
        pairwise["judge_id"] = judge_id
        pairwise["judge"] = JUDGE_LABELS[judge_id]
        pairwise_frames.append(pairwise)

availability = pd.DataFrame(availability_rows)


def aggregate_judge_values(source, group_column, metric_columns):
    rows = []
    for group_name, group in source.groupby(group_column, dropna=False):
        counts = pd.to_numeric(group["evaluated_examples"], errors="coerce")
        row = {
            group_column: group_name,
            "judge_count": group["judge_id"].nunique(),
            "evaluated_examples": counts.min(),
            "evaluated_examples_min": counts.min(),
            "evaluated_examples_max": counts.max(),
        }
        row["llm_judge_count"] = int(group.reindex(
            columns=[f"{metric}_mean" for metric in LLM_METRICS]
        ).notna().any(axis=1).sum())
        if "entity_recall_final_count" in group:
            row["recall_examples"] = pd.to_numeric(group["entity_recall_final_count"], errors="coerce").min()
        for column in metric_columns:
            values = pd.to_numeric(group[column], errors="coerce")
            row[column] = values.mean()
            row[column.removesuffix("_mean") + "_judge_count"] = int(values.notna().sum())
            row[column.removesuffix("_mean") + "_std"] = values.std(ddof=1)
        rows.append(row)
    return pd.DataFrame(rows)


judge_sources = {}
summaries = {}
for category in CATEGORY_NAMES:
    if not per_judge_summaries[category]:
        continue
    source = pd.concat(per_judge_summaries[category], ignore_index=True)
    metric_columns = [column for column in source.columns if column.endswith("_mean")]
    judge_sources[category] = source
    summaries[category] = aggregate_judge_values(source, "run", metric_columns)

if winrate_frames:
    winrate_source = pd.concat(winrate_frames, ignore_index=True)
    winrate_rows = []
    for method, group in winrate_source.groupby("method", dropna=False):
        row = {
            "method": method,
            "judge_count": group["judge_id"].nunique(),
            "comparisons_min": group["comparisons"].min(),
            "comparisons_max": group["comparisons"].max(),
        }
        for column in ["wins", "losses", "ties", "winrate"]:
            values = pd.to_numeric(group[column], errors="coerce")
            row[f"{column}_mean"] = values.mean()
            row[f"{column}_std"] = values.std(ddof=1)
        winrate_rows.append(row)
    summaries["answer_winrate"] = pd.DataFrame(winrate_rows)
else:
    winrate_source = pd.DataFrame()

if pairwise_frames:
    summaries["pairwise_answer_winrate"] = pd.concat(pairwise_frames, ignore_index=True)

display(availability)
summaries.keys()

,judge_id,judge,artifact,available,valid_llm_rows,llm_error_rows,path
0,gpt-5.4-mini,GPT-5.4 mini,bool_summary,True,273.0,0.0,evaluations/biomni-base/analysis/judges/gpt-5....
1,gpt-5.4-mini,GPT-5.4 mini,entity_summary,True,322.0,0.0,evaluations/biomni-base/analysis/judges/gpt-5....
2,gpt-5.4-mini,GPT-5.4 mini,numeric_summary,True,147.0,0.0,evaluations/biomni-base/analysis/judges/gpt-5....
3,gpt-5.4-mini,GPT-5.4 mini,answer_winrate_summary,True,NaN,NaN,evaluations/biomni-base/analysis/judges/gpt-5....
4,gpt-5.4-mini,GPT-5.4 mini,pairwise_answer_winrate,True,NaN,NaN,evaluations/biomni-base/analysis/judges/gpt-5....
5,qwen3.6-35b-a3b,Qwen 3.6 35B,bool_summary,True,271.0,2.0,evaluations/biomni-base/analysis/judges/qwen3....
6,qwen3.6-35b-a3b,Qwen 3.6 35B,entity_summary,True,319.0,3.0,evaluations/biomni-base/analysis/judges/qwen3....
7,qwen3.6-35b-a3b,Qwen 3.6 35B,numeric_summary,True,147.0,0.0,evaluations/biomni-base/analysis/judges/qwen3....
8,qwen3.6-35b-a3b,Qwen 3.6 35B,answer_winrate_summary,True,NaN,NaN,evaluations/biomni-base/analysis/judges/qwen3....
9,qwen3.6-35b-a3b,Qwen 3.6 35B,pairwise_answer_winrate,True,NaN,NaN,evaluations/biomni-base/analysis/judges/qwen3....


dict_keys(['bool', 'entity', 'numeric', 'answer_winrate', 'pairwise_answer_winrate'])

In [3]:
# Quick preview of each loaded summary.
for name, df in summaries.items():
    print(f"\n{name}: {df.shape[0]} rows x {df.shape[1]} columns")
    display(df.head())


bool: 7 rows x 51 columns


,run,judge_count,evaluated_examples,evaluated_examples_min,evaluated_examples_max,llm_judge_count,answer_token_precision_mean,answer_token_precision_judge_count,answer_token_precision_std,answer_token_recall_mean,answer_token_recall_judge_count,answer_token_recall_std,answer_token_f1_mean,answer_token_f1_judge_count,answer_token_f1_std,gt_entity_total_mean,gt_entity_total_judge_count,gt_entity_total_std,gt_entity_covered_mean,gt_entity_covered_judge_count,gt_entity_covered_std,gt_entity_coverage_mean,gt_entity_coverage_judge_count,gt_entity_coverage_std,bertscore_precision_mean,bertscore_precision_judge_count,bertscore_precision_std,bertscore_recall_mean,bertscore_recall_judge_count,bertscore_recall_std,bertscore_f1_mean,bertscore_f1_judge_count,bertscore_f1_std,llm_completeness_mean,llm_completeness_judge_count,llm_completeness_std,llm_faithfulness_mean,llm_faithfulness_judge_count,llm_faithfulness_std,llm_relevance_mean,llm_relevance_judge_count,llm_relevance_std,llm_understanderbility_mean,llm_understanderbility_judge_count,llm_understanderbility_std,nli_entailment_max_mean,nli_entailment_max_judge_count,nli_entailment_max_std,bool_accuracy_mean,bool_accuracy_judge_count,bool_accuracy_std
0,fullcontext,3,39,39,39,3,0.561577,3,0.000000e+00,0.501599,3,0.0,0.507475,3,0.0,1.769231,3,0.0,0.333333,3,0.0,0.117521,3,0.000000e+00,0.825013,3,0.0,0.856685,3,1.359740e-16,0.840080,3,0.000000e+00,0.094872,3,0.036352,0.115385,3,0.060624,0.709402,3,0.134569,0.976752,3,0.040266,0.597792,3,0.0,0.333333,3,0.000000e+00
1,grasp,3,39,39,39,3,0.217301,3,3.399350e-17,0.144743,3,0.0,0.159550,3,0.0,1.769231,3,0.0,0.153846,3,0.0,0.059829,3,8.498375e-18,0.832710,3,0.0,0.777822,3,1.359740e-16,0.802363,3,1.359740e-16,0.323999,3,0.099524,0.293837,3,0.014400,0.783311,3,0.058336,0.812123,3,0.081310,0.311397,3,0.0,0.384615,3,0.000000e+00
2,hipporag,3,39,39,39,3,0.082919,3,0.000000e+00,0.036299,3,0.0,0.044028,3,0.0,1.769231,3,0.0,0.410256,3,0.0,0.145299,3,0.000000e+00,0.727483,3,0.0,0.784602,3,0.000000e+00,0.754838,3,0.000000e+00,0.400247,3,0.089005,0.440711,3,0.031053,0.966082,3,0.034214,0.991228,3,0.015193,0.278777,3,0.0,0.487179,3,6.798700e-17
3,hypergraphrag,3,39,39,39,3,0.304457,3,0.000000e+00,0.687661,3,0.0,0.405765,3,0.0,1.769231,3,0.0,0.410256,3,0.0,0.164530,3,0.000000e+00,0.832518,3,0.0,0.861861,3,0.000000e+00,0.846417,3,0.000000e+00,0.596154,3,0.016960,0.585043,3,0.015824,0.900256,3,0.019784,0.948034,3,0.052574,0.477832,3,0.0,0.538462,3,0.000000e+00
4,llmbased,3,39,39,39,3,0.211613,3,0.000000e+00,0.326088,3,0.0,0.249819,3,0.0,1.769231,3,0.0,0.384615,3,0.0,0.138889,3,0.000000e+00,0.780733,3,0.0,0.802996,3,1.359740e-16,0.791610,3,0.000000e+00,0.046154,3,0.062229,0.073504,3,0.068435,0.230769,3,0.108453,0.617094,3,0.196390,0.164555,3,0.0,0.000000,3,0.000000e+00



entity: 7 rows x 76 columns


,run,judge_count,evaluated_examples,evaluated_examples_min,evaluated_examples_max,llm_judge_count,recall_examples,answer_token_precision_mean,answer_token_precision_judge_count,answer_token_precision_std,answer_token_recall_mean,answer_token_recall_judge_count,answer_token_recall_std,answer_token_f1_mean,answer_token_f1_judge_count,answer_token_f1_std,gt_entity_total_mean,gt_entity_total_judge_count,gt_entity_total_std,gt_entity_covered_mean,gt_entity_covered_judge_count,gt_entity_covered_std,gt_entity_coverage_mean,gt_entity_coverage_judge_count,gt_entity_coverage_std,bertscore_precision_mean,bertscore_precision_judge_count,bertscore_precision_std,bertscore_recall_mean,bertscore_recall_judge_count,bertscore_recall_std,bertscore_f1_mean,bertscore_f1_judge_count,bertscore_f1_std,llm_completeness_mean,llm_completeness_judge_count,llm_completeness_std,llm_faithfulness_mean,llm_faithfulness_judge_count,llm_faithfulness_std,llm_relevance_mean,llm_relevance_judge_count,llm_relevance_std,llm_understanderbility_mean,llm_understanderbility_judge_count,llm_understanderbility_std,nli_entailment_max_mean,nli_entailment_max_judge_count,nli_entailment_max_std,entity_gt_total_mean,entity_gt_total_judge_count,entity_gt_total_std,entity_retrieved_final_total_mean,entity_retrieved_final_total_judge_count,entity_retrieved_final_total_std,entity_retrieved_total_total_mean,entity_retrieved_total_total_judge_count,entity_retrieved_total_total_std,entity_recall_final_mean,entity_recall_final_judge_count,entity_recall_final_std,entity_precision_final_mean,entity_precision_final_judge_count,entity_precision_final_std,entity_f1_final_mean,entity_f1_final_judge_count,entity_f1_final_std,entity_recall_total_mean,entity_recall_total_judge_count,entity_recall_total_std,entity_precision_total_mean,entity_precision_total_judge_count,entity_precision_total_std,entity_f1_total_mean,entity_f1_total_judge_count,entity_f1_total_std
0,fullcontext,3,46,46,46,3,44,0.375428,3,6.798700e-17,0.237902,3,0.000000e+00,0.233184,3,0.0,4.978261,3,0.0,0.347826,3,0.0,0.111237,3,0.000000e+00,0.805008,3,0.000000e+00,0.812383,3,0.000000e+00,0.808287,3,0.0,0.041304,3,0.024571,0.073913,3,0.027755,0.403623,3,0.168311,0.958913,3,0.041529,0.441166,3,0.000000e+00,4.826087,3,0.0,4.804348,3,0.0,5.847826,3,1.087792e-15,0.000000,3,0.0,0.000000,3,0.000000e+00,NaN,0,NaN,0.000000,3,0.0,0.000000,3,0.000000e+00,NaN,0,NaN
1,grasp,3,46,46,46,3,44,0.276368,3,0.000000e+00,0.070897,3,0.000000e+00,0.098111,3,0.0,4.978261,3,0.0,0.065217,3,0.0,0.022727,3,4.249187e-18,0.831964,3,0.000000e+00,0.789693,3,1.359740e-16,0.809511,3,0.0,0.067971,3,0.010916,0.111828,3,0.031289,0.284968,3,0.135353,0.630588,3,0.235722,0.566474,3,0.000000e+00,4.826087,3,0.0,0.000000,3,0.0,0.000000,3,0.000000e+00,0.000000,3,0.0,NaN,0,NaN,NaN,0,NaN,0.000000,3,0.0,NaN,0,NaN,NaN,0,NaN
2,hipporag,3,46,46,46,3,44,0.450032,3,6.798700e-17,0.164688,3,0.000000e+00,0.137284,3,0.0,4.978261,3,0.0,2.347826,3,0.0,0.378093,3,0.000000e+00,0.751936,3,1.359740e-16,0.811002,3,0.000000e+00,0.779953,3,0.0,0.237174,3,0.059099,0.302391,3,0.041987,0.530217,3,0.148301,0.846522,3,0.150078,0.462035,3,0.000000e+00,4.826087,3,0.0,101.347826,3,0.0,252.586957,3,0.000000e+00,0.273611,3,0.0,0.012289,3,0.000000e+00,0.045944,3,0.0,0.730808,3,0.0,0.010682,3,0.000000e+00,0.022297,3,0.0
3,hypergraphrag,3,46,46,46,3,44,0.299865,3,0.000000e+00,0.372397,3,6.798700e-17,0.246984,3,0.0,4.978261,3,0.0,1.086957,3,0.0,0.270076,3,0.000000e+00,0.811944,3,0.000000e+00,0.820537,3,0.000000e+00,0.815499,3,0.0,0.424678,3,0.121592,0.444340,3,0.099573,0.737190,3,0.154388,0.916353,3,0.077614,0.448343,3,0.000000e+00,4.826087,3,0.0,0.000000,3,0.0,0.000000,3,0.000000e+00,0.000000,3,0.0,NaN,0,NaN,NaN,0,NaN,0.000000,3,0.0,NaN,0,NaN,NaN,0,NaN
4,llmbased,3,46,46,46,3,44,0.421169,3,0.000000e+00,0.284986,3,0.000000e+00,0.288454,3,0.0,4.978261,3,0.0,0.782609,3,0.0,0.160227,3,0.000000e+00,0.799983,3,0.000000e+00,0.811319,3,0.000000e+00,0.805241,3,0.0,0.125236,3,0.020749,0.133333,3,0.014758,0.357101,


numeric: 7 rows x 57 columns


,run,judge_count,evaluated_examples,evaluated_examples_min,evaluated_examples_max,llm_judge_count,answer_token_precision_mean,answer_token_precision_judge_count,answer_token_precision_std,answer_token_recall_mean,answer_token_recall_judge_count,answer_token_recall_std,answer_token_f1_mean,answer_token_f1_judge_count,answer_token_f1_std,gt_entity_total_mean,gt_entity_total_judge_count,gt_entity_total_std,gt_entity_covered_mean,gt_entity_covered_judge_count,gt_entity_covered_std,gt_entity_coverage_mean,gt_entity_coverage_judge_count,gt_entity_coverage_std,bertscore_precision_mean,bertscore_precision_judge_count,bertscore_precision_std,bertscore_recall_mean,bertscore_recall_judge_count,bertscore_recall_std,bertscore_f1_mean,bertscore_f1_judge_count,bertscore_f1_std,llm_completeness_mean,llm_completeness_judge_count,llm_completeness_std,llm_faithfulness_mean,llm_faithfulness_judge_count,llm_faithfulness_std,llm_relevance_mean,llm_relevance_judge_count,llm_relevance_std,llm_understanderbility_mean,llm_understanderbility_judge_count,llm_understanderbility_std,nli_entailment_max_mean,nli_entailment_max_judge_count,nli_entailment_max_std,numeric_ground_truth_count_mean,numeric_ground_truth_count_judge_count,numeric_ground_truth_count_std,numeric_predicted_count_mean,numeric_predicted_count_judge_count,numeric_predicted_count_std,numeric_accuracy_mean,numeric_accuracy_judge_count,numeric_accuracy_std
0,fullcontext,3,21,21,21,3,0.485783,3,6.798700e-17,0.444218,3,0.0,0.458992,3,0.0,1.52381,3,0.0,0.000000,3,0.000000e+00,0.000000,3,0.000000e+00,0.830512,3,1.359740e-16,0.877141,3,0.000000e+00,0.852858,3,1.359740e-16,0.015873,3,0.027493,0.082540,3,0.076980,0.500000,3,0.209145,0.985079,3,0.025843,0.732726,3,0.0,1.52381,3,0.0,7.153846,3,0.0,0.000000,3,0.000000e+00
1,grasp,3,21,21,21,3,0.219303,3,0.000000e+00,0.126313,3,0.0,0.120454,3,0.0,1.52381,3,0.0,0.000000,3,0.000000e+00,0.000000,3,0.000000e+00,0.818294,3,0.000000e+00,0.791153,3,1.359740e-16,0.802189,3,0.000000e+00,0.293651,3,0.013746,0.336508,3,0.087977,0.350794,3,0.010997,0.653968,3,0.231838,0.645753,3,0.0,1.52381,3,0.0,254.500000,3,0.0,0.190476,3,3.399350e-17
2,hipporag,3,21,21,21,3,0.842986,3,0.000000e+00,0.060895,3,0.0,0.086942,3,0.0,1.52381,3,0.0,0.000000,3,0.000000e+00,0.000000,3,0.000000e+00,0.715124,3,0.000000e+00,0.767079,3,0.000000e+00,0.739899,3,0.000000e+00,0.190476,3,0.062994,0.155556,3,0.021994,0.780952,3,0.082616,0.933016,3,0.025023,0.404436,3,0.0,1.52381,3,0.0,142.052632,3,0.0,0.142857,3,0.000000e+00
3,hypergraphrag,3,21,21,21,3,0.464553,3,6.798700e-17,0.585086,3,0.0,0.500181,3,0.0,1.52381,3,0.0,0.190476,3,3.399350e-17,0.190476,3,3.399350e-17,0.849935,3,1.359740e-16,0.878390,3,0.000000e+00,0.863190,3,0.000000e+00,0.497725,3,0.019330,0.471376,3,0.035878,0.738095,3,0.074231,0.973651,3,0.023061,0.565253,3,0.0,1.52381,3,0.0,6.000000,3,0.0,0.095238,3,1.699675e-17
4,llmbased,3,21,21,21,3,0.333005,3,0.000000e+00,0.374226,3,0.0,0.317195,3,0.0,1.52381,3,0.0,0.000000,3,0.000000e+00,0.000000,3,0.000000e+00,0.789269,3,0.000000e+00,0.797059,3,0.000000e+00,0.793017,3,1.359740e-16,0.025397,3,0.023490,0.070635,3,0.085061,0.134921,3,0.093597,0.715079,3,0.175390,0.489231,3,0.0,1.52381,3,0.0,5.050000,3,0.0,0.142857,3,0.000000e+00



answer_winrate: 7 rows x 12 columns


,method,judge_count,comparisons_min,comparisons_max,wins_mean,wins_std,losses_mean,losses_std,ties_mean,ties_std,winrate_mean,winrate_std
0,LWE,3,573,636,497.333333,42.500980,100.333333,27.465129,16.666667,20.428738,0.823339,0.033149
1,fullcontext,3,96,106,16.333333,7.767453,86.333333,8.386497,0.000000,0.000000,0.158674,0.071623
2,grasp,3,95,106,13.000000,6.000000,86.666667,6.027714,2.666667,3.785939,0.139043,0.039370
3,hipporag,3,98,106,10.666667,6.429101,89.000000,8.185353,3.666667,5.507571,0.120748,0.050073
4,hypergraphrag,3,92,106,33.000000,7.000000,62.333333,12.013881,5.666667,3.511885,0.357223,0.072938



pairwise_answer_winrate: 1908 rows x 20 columns


,evaluation_judge_id,evaluation_config_path,ground_truth_id,ground_truth_qtype,question,ground_truth_answer,method_a,method_b,method_a_key,method_b_key,answer_a,answer_b,winner,judge_rationale,winning_method,method_a_score,method_b_score,judge_id,judge,judge_error
0,gpt-5.4-mini,/home/desild/work/research/LLM-Workflow-Explor...,gt_0,"[""multi"", ""numeric""]","\nHow many ""experiment execution"" are there in...",The answer to the question is 12 unique execut...,fullcontext,LWE,fullcontext,ours,There are 16 distinct experiment executions re...,The provenance document lists two distinct exe...,method_a,The ground truth says there are 12 unique exec...,fullcontext,1.0,0.0,gpt-5.4-mini,GPT-5.4 mini,NaN
1,gpt-5.4-mini,/home/desild/work/research/LLM-Workflow-Explor...,gt_1,"[""multi"", ""entity""]",\nIn what places do we utilize AI in this work...,\nThe ChatBS System utilizes AI for the follow...,fullcontext,LWE,fullcontext,ours,The workflow uses AI in four distinct stages w...,In this Biomni workflow the artificial‑intelli...,method_b,"The ground truth says AI is used at steps 4, 5...",LWE,0.0,1.0,gpt-5.4-mini,GPT-5.4 mini,NaN
2,gpt-5.4-mini,/home/desild/work/research/LLM-Workflow-Explor...,gt_10,"[""multi"", ""entity""]",\nwhat are the output parameters of the functi...,\n The input parameters to Step 6 - Biomni ...,fullcontext,LWE,fullcontext,ours,The function used in Step 6 – Biomni LLM gener...,The program that implements **Step 6 – Biomni ...,method_b,Method B is closer to the ground truth because...,LWE,0.0,1.0,gpt-5.4-mini,GPT-5.4 mini,NaN
3,gpt-5.4-mini,/home/desild/work/research/LLM-Workflow-Explor...,gt_100,"[""multi"", ""numeric""]","\nwhat is the number of output for the ""Step 6...","\n The number of output for the ""Step 6 - L...",fullcontext,LWE,fullcontext,ours,I cannot determine that number because the exe...,The execution identified by `http://testwebsit...,method_b,Method B matches the ground truth by stating t...,LWE,0.0,1.0,gpt-5.4-mini,GPT-5.4 mini,NaN
4,gpt-5.4-mini,/home/desild/work/research/LLM-Workflow-Explor...,gt_101,"[""multi"", ""numeric""]","\nwhat is the number of output for the ""Step 6...","\n The number of output for the ""Step 6 - r...",fullcontext,LWE,fullcontext,ours,The graph does not contain the execution with ...,The “Step 6 – response output port” in the exe...,method_b,Method B matches the ground truth by stating t...,LWE,0.0,1.0,gpt-5.4-mini,GPT-5.4 mini,NaN


## Inspect Available Columns

Use this cell when choosing the placeholders below.

In [4]:
for name, df in summaries.items():
    print(f"\n{name}")
    for column in df.columns:
        print(f"  - {column}")


bool
  - run
  - judge_count
  - evaluated_examples
  - evaluated_examples_min
  - evaluated_examples_max
  - llm_judge_count
  - answer_token_precision_mean
  - answer_token_precision_judge_count
  - answer_token_precision_std
  - answer_token_recall_mean
  - answer_token_recall_judge_count
  - answer_token_recall_std
  - answer_token_f1_mean
  - answer_token_f1_judge_count
  - answer_token_f1_std
  - gt_entity_total_mean
  - gt_entity_total_judge_count
  - gt_entity_total_std
  - gt_entity_covered_mean
  - gt_entity_covered_judge_count
  - gt_entity_covered_std
  - gt_entity_coverage_mean
  - gt_entity_coverage_judge_count
  - gt_entity_coverage_std
  - bertscore_precision_mean
  - bertscore_precision_judge_count
  - bertscore_precision_std
  - bertscore_recall_mean
  - bertscore_recall_judge_count
  - bertscore_recall_std
  - bertscore_f1_mean
  - bertscore_f1_judge_count
  - bertscore_f1_std
  - llm_completeness_mean
  - llm_completeness_judge_count
  - llm_completeness_std
  - ll

## Editable Paper Table Columns

Edit these lists to decide what goes into each paper table. Keep `run` or `method` first if you want method names visible.

In [5]:
# These columns show ensemble means and between-judge sample standard deviations.
bool_columns = [
    "run",
    "judge_count",
    "llm_judge_count",
    "evaluated_examples",
    "bertscore_f1_mean",
    "bertscore_f1_std",
    "nli_entailment_max_mean",
    "nli_entailment_max_std",
    "bool_accuracy_mean",
    "bool_accuracy_std",
]

entity_columns = [
    "run",
    "recall_examples",
    "judge_count",
    "llm_judge_count",
    "evaluated_examples",
    "answer_token_f1_mean",
    "answer_token_f1_std",
    "bertscore_f1_mean",
    "bertscore_f1_std",
    "nli_entailment_max_mean",
    "nli_entailment_max_std",
    "entity_recall_final_mean",
    "entity_recall_final_std",
    "entity_precision_final_mean",
    "entity_precision_final_std",
    "entity_f1_final_mean",
    "entity_f1_final_std",
    "entity_recall_total_mean",
    "entity_recall_total_std",
    "entity_precision_total_mean",
    "entity_precision_total_std",
    "entity_f1_total_mean",
    "entity_f1_total_std",
]

numeric_columns = [
    "run",
    "judge_count",
    "llm_judge_count",
    "evaluated_examples",
    "answer_token_f1_mean",
    "answer_token_f1_std",
    "numeric_accuracy_mean",
    "numeric_accuracy_std",
    "bertscore_f1_mean",
    "bertscore_f1_std",
    "nli_entailment_max_mean",
    "nli_entailment_max_std",
]

answer_winrate_columns = [
    "method",
    "judge_count",
    "comparisons_min",
    "comparisons_max",
    "winrate_mean",
    "winrate_std",
]

In [6]:
# Optional display names. Add, remove, or rename as needed for the paper.
column_labels = {
    "run": "Method",
    "method": "Method",
    "judge_count": "Result Sources",
    "llm_judge_count": "LLM Judges",
    "recall_examples": "Recall N",
    "evaluated_examples": "N",
    "evaluated_examples_min": "N Min",
    "evaluated_examples_max": "N Max",
    "comparisons_min": "Comparisons Min",
    "comparisons_max": "Comparisons Max",
    "answer_token_f1_mean": "Answer Token F1",
    "answer_token_f1_std": "Answer Token F1 SD",
    "bertscore_f1_mean": "BERTScore F1",
    "bertscore_f1_std": "BERTScore F1 SD",
    "nli_entailment_max_mean": "NLI Entailment",
    "nli_entailment_max_std": "NLI Entailment SD",
    "bool_accuracy_mean": "Bool Accuracy",
    "bool_accuracy_std": "Bool Accuracy SD",
    "entity_recall_final_mean": "Entity Recall",
    "entity_recall_final_std": "Entity Recall SD",
    "entity_precision_final_mean": "Entity Precision",
    "entity_precision_final_std": "Entity Precision SD",
    "entity_f1_final_mean": "Entity F1",
    "entity_f1_final_std": "Entity F1 SD",
    "entity_recall_total_mean": "Entity Recall Total",
    "entity_recall_total_std": "Entity Recall Total SD",
    "entity_precision_total_mean": "Entity Precision Total",
    "entity_precision_total_std": "Entity Precision Total SD",
    "entity_f1_total_mean": "Entity F1 Total",
    "entity_f1_total_std": "Entity F1 Total SD",
    "numeric_accuracy_mean": "Numeric Accuracy",
    "numeric_accuracy_std": "Numeric Accuracy SD",
    "llm_completeness_mean": "Completeness",
    "llm_completeness_std": "Completeness SD",
    "llm_completeness_judge_count": "Completeness Judges",
    "llm_relevance_mean": "Relevance",
    "llm_relevance_std": "Relevance SD",
    "llm_relevance_judge_count": "Relevance Judges",
    "llm_faithfulness_mean": "Faithfulness",
    "llm_faithfulness_std": "Faithfulness SD",
    "llm_faithfulness_judge_count": "Faithfulness Judges",
    "llm_understanderbility_mean": "Understandability",
    "llm_understanderbility_std": "Understandability SD",
    "llm_understanderbility_judge_count": "Understandability Judges",
    "winrate_mean": "Win Rate",
    "winrate_std": "Win Rate SD",
}

method_order = ["GWB", "VSB", "GRASP", "HippoRAG", "HyperGRAG", "Ours"]

method_labels = {
    "fullcontext": "FCB",
    "grasp": "GRASP",
    "hipporag": "HippoRAG",
    "hypergraphrag": "HyperGRAG",
    "llmbased": "GWB",
    "LWE": "Ours",
    "ours": "Ours",
    "vectorsimilarity": "VSB",
}

def order_method_rows(table, method_column="Method"):
    ordered = table[table[method_column].isin(method_order)].copy()
    ordered[method_column] = pd.Categorical(
        ordered[method_column],
        categories=method_order,
        ordered=True,
    )
    sort_columns = [method_column]
    if "Question Type" in ordered.columns:
        sort_columns = ["Question Type", method_column]
    return ordered.sort_values(sort_columns).reset_index(drop=True)

count_columns = {"Result Sources", "LLM Judges", "N", "N Min", "N Max", "Recall N", "Comparisons Min", "Comparisons Max"}

def scale_result_columns(table):
    scaled = table.copy()
    for position, column in enumerate(scaled.columns):
        if column in count_columns or column.endswith(" Judges") or column.endswith("_judge_count"):
            continue
        values = scaled.iloc[:, position]
        if pd.api.types.is_numeric_dtype(values):
            scaled.iloc[:, position] = values * 100
    return scaled

## Build Individual Tables

In [7]:
round_digits = 3

bool_table = summaries["bool"][bool_columns].copy()
bool_table["run"] = bool_table["run"].replace(method_labels)
bool_table = scale_result_columns(bool_table.rename(columns=column_labels)).round(round_digits)
bool_table = order_method_rows(bool_table)

entity_table = summaries["entity"][entity_columns].copy()
entity_table["run"] = entity_table["run"].replace(method_labels)
entity_table = scale_result_columns(entity_table.rename(columns=column_labels)).round(round_digits)
entity_table = order_method_rows(entity_table)

numeric_table = summaries["numeric"][numeric_columns].copy()
numeric_table["run"] = numeric_table["run"].replace(method_labels)
numeric_table = scale_result_columns(numeric_table.rename(columns=column_labels)).round(round_digits)
numeric_table = order_method_rows(numeric_table)



In [8]:
if "answer_winrate" in summaries:
    answer_winrate_table = summaries["answer_winrate"][answer_winrate_columns].copy()
    answer_winrate_table["method"] = answer_winrate_table["method"].replace(method_labels)
    answer_winrate_table = scale_result_columns(answer_winrate_table.rename(columns=column_labels)).round(round_digits)
    answer_winrate_table = order_method_rows(answer_winrate_table)
else:
    answer_winrate_table = pd.DataFrame(
        columns=[column_labels.get(column, column) for column in answer_winrate_columns]
    )

In [9]:
display(bool_table)
display(entity_table)
display(numeric_table)
display(answer_winrate_table)

,Method,Result Sources,LLM Judges,N,BERTScore F1,BERTScore F1 SD,NLI Entailment,NLI Entailment SD,Bool Accuracy,Bool Accuracy SD
0,GWB,3,3,39,79.161,0.0,16.456,0.0,0.000,0.0
1,VSB,3,3,39,80.637,0.0,40.477,0.0,43.590,0.0
2,GRASP,3,3,39,80.236,0.0,31.140,0.0,38.462,0.0
3,HippoRAG,3,3,39,75.484,0.0,27.878,0.0,48.718,0.0
4,HyperGRAG,3,3,39,84.642,0.0,47.783,0.0,53.846,0.0
5,Ours,3,3,39,82.982,0.0,45.537,0.0,76.923,0.0


,Method,Recall N,Result Sources,LLM Judges,N,Answer Token F1,Answer Token F1 SD,BERTScore F1,BERTScore F1 SD,NLI Entailment,NLI Entailment SD,Entity Recall,Entity Recall SD,Entity Precision,Entity Precision SD,Entity F1,Entity F1 SD,Entity Recall Total,Entity Recall Total SD,Entity Precision Total,Entity Precision Total SD,Entity F1 Total,Entity F1 Total SD
0,GWB,44,3,3,46,28.845,0.0,80.524,0.0,42.647,0.0,28.662,0.0,4.783,0.0,13.242,0.0,28.662,0.0,4.600,0.0,12.846,0.0
1,VSB,44,3,3,46,22.512,0.0,81.182,0.0,41.907,0.0,6.692,0.0,3.244,0.0,18.780,0.0,15.581,0.0,3.774,0.0,13.190,0.0
2,GRASP,44,3,3,46,9.811,0.0,80.951,0.0,56.647,0.0,0.000,0.0,NaN,NaN,NaN,NaN,0.000,0.0,NaN,NaN,NaN,NaN
3,HippoRAG,44,3,3,46,13.728,0.0,77.995,0.0,46.204,0.0,27.361,0.0,1.229,0.0,4.594,0.0,73.081,0.0,1.068,0.0,2.230,0.0
4,HyperGRAG,44,3,3,46,24.698,0.0,81.550,0.0,44.834,0.0,0.000,0.0,NaN,NaN,NaN,NaN,0.000,0.0,NaN,NaN,NaN,NaN
5,Ours,44,3,3,46,27.612,0.0,81.307,0.0,40.878,0.0,17.992,0.0,0.372,0.0,1.829,0.0,37.386,0.0,2.131,0.0,6.851,0.0


,Method,Result Sources,LLM Judges,N,Answer Token F1,Answer Token F1 SD,Numeric Accuracy,Numeric Accuracy SD,BERTScore F1,BERTScore F1 SD,NLI Entailment,NLI Entailment SD
0,GWB,3,3,21,31.720,0.0,14.286,0.0,79.302,0.0,48.923,0.0
1,VSB,3,3,21,18.169,0.0,0.000,0.0,78.821,0.0,49.384,0.0
2,GRASP,3,3,21,12.045,0.0,19.048,0.0,80.219,0.0,64.575,0.0
3,HippoRAG,3,3,21,8.694,0.0,14.286,0.0,73.990,0.0,40.444,0.0
4,HyperGRAG,3,3,21,50.018,0.0,9.524,0.0,86.319,0.0,56.525,0.0
5,Ours,3,3,21,48.821,0.0,52.381,0.0,86.289,0.0,55.705,0.0


,Method,Result Sources,Comparisons Min,Comparisons Max,Win Rate,Win Rate SD
0,GWB,3,92,106,8.911,5.357
1,VSB,3,100,106,19.752,5.597
2,GRASP,3,95,106,13.904,3.937
3,HippoRAG,3,98,106,12.075,5.007
4,HyperGRAG,3,92,106,35.722,7.294
5,Ours,3,573,636,82.334,3.315


## Combined Table Placeholder

This creates one compact table across question types. Edit `combined_columns` to choose the shared metrics.

In [10]:
combined_columns = [
    "task",
    "run",
    "judge_count",
    "evaluated_examples",
    "answer_token_f1_mean",
    "answer_token_f1_std",
    "bertscore_f1_mean",
    "bertscore_f1_std",
    "nli_entailment_max_mean",
    "nli_entailment_max_std",
]

combined_parts = []
for task_name in ["bool", "entity", "numeric"]:
    part = summaries[task_name].copy()
    part.insert(0, "task", task_name)
    combined_parts.append(part)

combined_table = pd.concat(combined_parts, ignore_index=True)
combined_table = combined_table[combined_columns].copy()
combined_table["run"] = combined_table["run"].replace(method_labels)
combined_table = scale_result_columns(combined_table.rename(columns={**column_labels, "task": "Question Type"})).round(round_digits)
combined_table = order_method_rows(combined_table)

display(combined_table)

,Question Type,Method,Result Sources,N,Answer Token F1,Answer Token F1 SD,BERTScore F1,BERTScore F1 SD,NLI Entailment,NLI Entailment SD
0,bool,GWB,3,39,24.982,0.0,79.161,0.0,16.456,0.0
1,bool,VSB,3,39,33.134,0.0,80.637,0.0,40.477,0.0
2,bool,GRASP,3,39,15.955,0.0,80.236,0.0,31.140,0.0
3,bool,HippoRAG,3,39,4.403,0.0,75.484,0.0,27.878,0.0
4,bool,HyperGRAG,3,39,40.576,0.0,84.642,0.0,47.783,0.0
5,bool,Ours,3,39,38.079,0.0,82.982,0.0,45.537,0.0
6,entity,GWB,3,46,28.845,0.0,80.524,0.0,42.647,0.0
7,entity,VSB,3,46,22.512,0.0,81.182,0.0,41.907,0.0
8,entity,GRASP,3,46,9.811,0.0,80.951,0.0,56.647,0.0
9,entity,HippoRAG,3,46,13.728,0.0,77.995,0.0,46.204,0.0


## Overall Summary Table

LLM metrics use valid item-level scores across all question types within each judge. Recall uses entity questions only. Mean and sample SD are calculated across available judge means. Result Sources counts result sets, LLM Judges counts judges with quality scores, and each attribute also has its own valid judge count. A requested judge with no scores is missing, not a zero score.


In [11]:
overall_metric_names = [
    "answer_token_f1", "bertscore_f1", "nli_entailment_max",
    *LLM_METRICS, "entity_recall_final", "entity_recall_total",
]
overall_parts = []
for judge_id in JUDGE_IDS:
    for category in CATEGORY_NAMES:
        frame = read_csv_or_empty(judge_analysis_dir(judge_id) / category / "results.csv")
        if not frame.empty:
            overall_parts.append(frame.assign(judge_id=judge_id, question_type=category))
overall_source = pd.concat(overall_parts, ignore_index=True)
if overall_source.duplicated(["judge_id", "run", "ground_truth_id", "question_type"]).any():
    raise ValueError("Duplicate per-question scores in overall summary")
overall_rows = []
for (judge_id, run_name), group in overall_source.groupby(["judge_id", "run"]):
    row = {"judge_id": judge_id, "run": run_name, "evaluated_examples": len(group)}
    for metric in overall_metric_names:
        metric_rows = group[group["question_type"].eq("entity")] if metric.startswith("entity_recall_") else group
        values = pd.to_numeric(
            metric_rows.get(metric, pd.Series(np.nan, index=metric_rows.index)),
            errors="coerce",
        )
        row[f"{metric}_mean"] = values.mean()
        row[f"{metric}_count"] = int(values.notna().sum())
    row["recall_examples"] = row["entity_recall_final_count"]
    overall_rows.append(row)
overall_judge_source = pd.DataFrame(overall_rows)
overall_summary_table = aggregate_judge_values(
    overall_judge_source, "run", [f"{metric}_mean" for metric in overall_metric_names],
)
overall_summary_table["run"] = overall_summary_table["run"].replace(method_labels)
overall_summary_table = scale_result_columns(overall_summary_table.rename(columns=column_labels)).round(round_digits)
overall_summary_table = order_method_rows(overall_summary_table)
display(overall_summary_table)
if overall_summary_table["LLM Judges"].lt(len(JUDGE_IDS)).any():
    print("Incomplete LLM ensemble: means and SD use available ratings. See the per-attribute judge counts and availability tables.")


,Method,Result Sources,N,N Min,N Max,LLM Judges,Recall N,Answer Token F1,answer_token_f1_judge_count,Answer Token F1 SD,BERTScore F1,bertscore_f1_judge_count,BERTScore F1 SD,NLI Entailment,nli_entailment_max_judge_count,NLI Entailment SD,Completeness,Completeness Judges,Completeness SD,Relevance,Relevance Judges,Relevance SD,Faithfulness,Faithfulness Judges,Faithfulness SD,Understandability,Understandability Judges,Understandability SD,Entity Recall,entity_recall_final_judge_count,Entity Recall SD,Entity Recall Total,entity_recall_total_judge_count,Entity Recall Total SD
0,GWB,3,106,106,106,3,44,27.993,3,0.0,79.780,3,0.0,34.254,3,0.0,7.636,3,3.061,26.660,3,9.184,9.890,3,4.333,64.016,3,16.664,28.662,3,0.0,28.662,3,0.0
1,VSB,3,106,106,106,3,44,25.560,3,0.0,80.514,3,0.0,42.862,3,0.0,22.981,3,2.459,63.959,3,13.892,30.025,3,4.171,96.673,3,2.562,6.692,3,0.0,15.581,3,0.0
2,GRASP,3,106,106,106,3,44,12.514,3,0.0,80.488,3,0.0,45.263,3,0.0,20.699,3,3.161,48.103,3,8.079,22.340,3,3.512,70.175,3,17.699,0.000,3,0.0,0.000,3,0.0
3,HippoRAG,3,106,106,106,3,44,9.300,3,0.0,76.278,3,0.0,38.320,3,0.0,28.775,3,3.274,73.957,3,6.846,32.390,3,1.932,91.645,3,7.633,27.361,3,0.0,73.081,3,0.0
4,HyperGRAG,3,106,106,106,3,44,35.557,3,0.0,83.632,3,0.0,48.235,3,0.0,50.225,3,5.572,79.739,3,7.024,50.140,3,3.257,93.949,3,5.706,0.000,3,0.0,0.000,3,0.0
5,Ours,3,106,106,106,3,44,35.665,3,0.0,82.911,3,0.0,45.530,3,0.0,68.454,3,6.102,93.748,3,8.116,65.469,3,8.492,90.819,3,9.148,17.992,3,0.0,37.386,3,0.0


## Export Tables

Exports CSV and LaTeX versions into `paper_tables/`. Comment out any table you do not need.

In [12]:
tables = {
    "bool_summary": bool_table,
    "entity_summary": entity_table,
    "numeric_summary": numeric_table,
    "answer_winrate_summary": answer_winrate_table,
    "combined_summary": combined_table,
    "overall_summary": overall_summary_table,
}

for table_name, table in tables.items():
    csv_path = TABLE_OUTPUT_DIR / f"{table_name}.csv"
    tex_path = TABLE_OUTPUT_DIR / f"{table_name}.tex"

    table.to_csv(csv_path, index=False)
    table.to_latex(tex_path, index=False, escape=False)

    print(f"Wrote {csv_path}")
    print(f"Wrote {tex_path}")



availability.to_csv(TABLE_OUTPUT_DIR / "judge_artifact_availability.csv", index=False)
for category, source in judge_sources.items():
    source.to_csv(TABLE_OUTPUT_DIR / f"{category}_summary_by_judge.csv", index=False)
overall_judge_source.to_csv(TABLE_OUTPUT_DIR / "overall_summary_by_judge.csv", index=False)
if not winrate_source.empty:
    winrate_source.to_csv(TABLE_OUTPUT_DIR / "answer_winrate_summary_by_judge.csv", index=False)
if "pairwise_answer_winrate" in summaries:
    summaries["pairwise_answer_winrate"].to_csv(TABLE_OUTPUT_DIR / "pairwise_answer_winrate_by_judge.csv", index=False)


Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/biomni-base/paper_tables/judges/ensemble/bool_summary.csv
Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/biomni-base/paper_tables/judges/ensemble/bool_summary.tex
Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/biomni-base/paper_tables/judges/ensemble/entity_summary.csv
Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/biomni-base/paper_tables/judges/ensemble/entity_summary.tex
Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/biomni-base/paper_tables/judges/ensemble/numeric_summary.csv
Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/biomni-base/paper_tables/judges/ensemble/numeric_summary.tex
Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/biomni-base/paper_tables/judges/ensemble/answer_winrate_summary.csv
Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/biomni-base/paper_tables/judges/ensemb

## Optional: Pairwise Win Rate Details

In [13]:
if "pairwise_answer_winrate" in summaries:
    pairwise_table = summaries["pairwise_answer_winrate"].copy()
    print(f"Loaded {len(pairwise_table)} pairwise decisions across {pairwise_table['judge_id'].nunique()} judges")
    preview_columns = [
        column for column in [
            "judge",
            "ground_truth_id",
            "method_a_key",
            "method_b_key",
            "winning_method",
        ]
        if column in pairwise_table.columns
    ]
    display(pairwise_table[preview_columns].head(20))

Loaded 1908 pairwise decisions across 3 judges


,judge,ground_truth_id,method_a_key,method_b_key,winning_method
0,GPT-5.4 mini,gt_0,fullcontext,ours,fullcontext
1,GPT-5.4 mini,gt_1,fullcontext,ours,LWE
2,GPT-5.4 mini,gt_10,fullcontext,ours,LWE
3,GPT-5.4 mini,gt_100,fullcontext,ours,LWE
4,GPT-5.4 mini,gt_101,fullcontext,ours,LWE
5,GPT-5.4 mini,gt_102,fullcontext,ours,LWE
6,GPT-5.4 mini,gt_103,fullcontext,ours,LWE
7,GPT-5.4 mini,gt_104,fullcontext,ours,LWE
8,GPT-5.4 mini,gt_105,fullcontext,ours,LWE
9,GPT-5.4 mini,gt_11,fullcontext,ours,LWE


## Krippendorff Alpha Agreement

This final section measures how consistently the LLM judges assign the **completeness**, **relevance**, and **faithfulness** scores used in the paper tables. It computes **interval Krippendorff's alpha** directly from the item-level `[0, 1]` ratings in each judge's raw `results.csv`; squared-distance disagreement is appropriate because the attributes are numeric rather than unordered labels.

A rating item is aligned by question type, method, and ground-truth ID. The overall rows cover the same six methods retained in the paper tables, and the remaining rows show agreement for each method separately. Items with at least two available ratings are usable, so an absent, failed, or partially completed judge run does not prevent calculation. Alpha is `1` for perfect agreement, approximately `0` when agreement is no better than expected from the pooled ratings, and can be negative for systematic disagreement. This item-level reliability measure complements the between-judge standard deviations of the aggregate means above.

Only GPT, Qwen, and Gemma are selected. Judges without valid ratings do not contribute to alpha; the output distinguishes usable Judges from Requested Judges. Llama is excluded.


In [14]:
AGREEMENT_ATTRIBUTES = {
    "llm_completeness": "Completeness",
    "llm_relevance": "Relevance",
    "llm_faithfulness": "Faithfulness",
}
AGREEMENT_METHODS = [
    "llmbased",
    "vectorsimilarity",
    "grasp",
    "hipporag",
    "hypergraphrag",
    "ours",
]
ITEM_COLUMNS = ["question_type", "run", "ground_truth_id"]


def load_item_level_judge_ratings():
    frames = []
    rows = []
    for judge_id in JUDGE_IDS:
        for question_type in CATEGORY_NAMES:
            path = judge_analysis_dir(judge_id) / question_type / "results.csv"
            frame = read_csv_or_empty(path)
            available_attributes = [column for column in AGREEMENT_ATTRIBUTES if column in frame.columns]
            usable = (
                not frame.empty and {"run", "ground_truth_id"}.issubset(frame.columns)
                and bool(available_attributes)
                and frame[available_attributes].apply(pd.to_numeric, errors="coerce").notna().any().any()
            )
            rows.append({
                "judge_id": judge_id,
                "judge": JUDGE_LABELS[judge_id],
                "question_type": question_type,
                "available": usable,
                "rows": len(frame),
                "attributes": ", ".join(available_attributes),
                "path": str(path.relative_to(REPO_ROOT)),
            })
            if not usable or not available_attributes:
                continue
            selected = frame[["run", "ground_truth_id", *available_attributes]].copy()
            selected.insert(0, "question_type", question_type)
            selected.insert(0, "judge", JUDGE_LABELS[judge_id])
            selected.insert(0, "judge_id", judge_id)
            frames.append(selected)
    source = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    return source, pd.DataFrame(rows)


def krippendorff_alpha_interval(ratings):
    numeric = ratings.apply(pd.to_numeric, errors="coerce")
    usable = numeric.loc[numeric.notna().sum(axis=1).ge(2)]
    if usable.empty:
        return None
    observed_numerator = 0.0
    pooled_ratings = []
    for _, item_ratings in usable.iterrows():
        values = item_ratings.dropna().to_numpy(dtype=float)
        rating_count = len(values)
        centered_sum_squares = float(((values - values.mean()) ** 2).sum())
        observed_numerator += 2 * rating_count * centered_sum_squares / (rating_count - 1)
        pooled_ratings.extend(values)
    pooled = np.asarray(pooled_ratings, dtype=float)
    ratings_per_item = usable.notna().sum(axis=1)
    total_ratings = len(pooled)
    observed_disagreement = observed_numerator / total_ratings
    expected_disagreement = 2 * float(((pooled - pooled.mean()) ** 2).sum()) / (total_ratings - 1)
    alpha = 1 - observed_disagreement / expected_disagreement if expected_disagreement > 0 else np.nan
    return {
        "items_total": len(numeric),
        "items_used": len(usable),
        "ratings_used": total_ratings,
        "min_ratings_per_item": int(ratings_per_item.min()),
        "max_ratings_per_item": int(ratings_per_item.max()),
        "observed_disagreement": observed_disagreement,
        "expected_disagreement": expected_disagreement,
        "krippendorff_alpha": alpha,
    }


agreement_source, agreement_availability = load_item_level_judge_ratings()
alpha_rows = []
if not agreement_source.empty:
    agreement_source["ground_truth_id"] = agreement_source["ground_truth_id"].astype(str)
    agreement_source = agreement_source[agreement_source["run"].isin(AGREEMENT_METHODS)].copy()
    if agreement_source.duplicated(["judge_id", *ITEM_COLUMNS]).any():
        raise ValueError("Duplicate attribute ratings found for the same judge and item")

    for attribute, attribute_label in AGREEMENT_ATTRIBUTES.items():
        if attribute not in agreement_source.columns:
            continue
        rating_matrix = agreement_source.pivot(
            index=ITEM_COLUMNS,
            columns="judge_id",
            values=attribute,
        )
        scopes = [("overall", "all", "All paper methods", rating_matrix)]
        scopes.extend(
            (
                "method",
                method,
                method_labels.get(method, method),
                rating_matrix[rating_matrix.index.get_level_values("run") == method],
            )
            for method in AGREEMENT_METHODS
        )
        for scope_type, scope, scope_label, scoped_matrix in scopes:
            scope_judges = [
                judge_id for judge_id in JUDGE_IDS
                if judge_id in scoped_matrix.columns and scoped_matrix[judge_id].notna().any()
            ]
            stats = krippendorff_alpha_interval(scoped_matrix[scope_judges])
            if stats is not None:
                alpha_rows.append({
                    "attribute": attribute_label,
                    "scope_type": scope_type,
                    "scope": scope,
                    "scope_label": scope_label,
                    "distance": "interval_squared",
                    "judge_count": len(scope_judges),
                    "requested_judge_count": len(JUDGE_IDS),
                    "judges": ", ".join(scope_judges),
                    **stats,
                })

krippendorff_agreement = pd.DataFrame(alpha_rows)
alpha_path = TABLE_OUTPUT_DIR / "krippendorff_alpha_agreement.csv"
availability_path = TABLE_OUTPUT_DIR / "krippendorff_alpha_input_availability.csv"
krippendorff_agreement.to_csv(alpha_path, index=False)
agreement_availability.to_csv(availability_path, index=False)
print(f"Wrote {alpha_path}")
print(f"Wrote {availability_path}")
if krippendorff_agreement.empty:
    print("At least two judges with overlapping item-level attribute ratings are required for alpha.")
else:
    alpha_overall_table = krippendorff_agreement.loc[
        krippendorff_agreement["scope_type"].eq("overall"),
        ["attribute", "judge_count", "requested_judge_count", "items_used", "ratings_used", "krippendorff_alpha"],
    ].rename(columns={
        "attribute": "Attribute",
        "judge_count": "Judges",
        "requested_judge_count": "Requested Judges",
        "items_used": "Items",
        "ratings_used": "Ratings",
        "krippendorff_alpha": "Krippendorff Alpha",
    })
    overall_csv_path = TABLE_OUTPUT_DIR / "krippendorff_alpha_overall.csv"
    overall_tex_path = TABLE_OUTPUT_DIR / "krippendorff_alpha_overall.tex"
    alpha_overall_table.round(3).to_csv(overall_csv_path, index=False)
    alpha_overall_table.round(3).to_latex(overall_tex_path, index=False, escape=False)
    print(f"Wrote {overall_csv_path}")
    print(f"Wrote {overall_tex_path}")
    display(alpha_overall_table.round(3))
    display(krippendorff_agreement.loc[krippendorff_agreement["scope_type"].eq("method")].round(3))

Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/biomni-base/paper_tables/judges/ensemble/krippendorff_alpha_agreement.csv
Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/biomni-base/paper_tables/judges/ensemble/krippendorff_alpha_input_availability.csv
Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/biomni-base/paper_tables/judges/ensemble/krippendorff_alpha_overall.csv
Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/biomni-base/paper_tables/judges/ensemble/krippendorff_alpha_overall.tex


,Attribute,Judges,Requested Judges,Items,Ratings,Krippendorff Alpha
0,Completeness,3,3,636,1903,0.841
7,Relevance,3,3,636,1903,0.719
14,Faithfulness,3,3,636,1903,0.771


,attribute,scope_type,scope,scope_label,distance,judge_count,requested_judge_count,judges,items_total,items_used,ratings_used,min_ratings_per_item,max_ratings_per_item,observed_disagreement,expected_disagreement,krippendorff_alpha
1,Completeness,method,llmbased,GWB,interval_squared,3,3,"gpt-5.4-mini, qwen3.6-35b-a3b, gemma-4-31b",106,106,318,3,3,0.012,0.035,0.646
2,Completeness,method,vectorsimilarity,VSB,interval_squared,3,3,"gpt-5.4-mini, qwen3.6-35b-a3b, gemma-4-31b",106,106,318,3,3,0.054,0.268,0.800
3,Completeness,method,grasp,GRASP,interval_squared,3,3,"gpt-5.4-mini, qwen3.6-35b-a3b, gemma-4-31b",106,106,316,2,3,0.057,0.292,0.803
4,Completeness,method,hipporag,HippoRAG,interval_squared,3,3,"gpt-5.4-mini, qwen3.6-35b-a3b, gemma-4-31b",106,106,317,2,3,0.069,0.313,0.779
5,Completeness,method,hypergraphrag,HyperGRAG,interval_squared,3,3,"gpt-5.4-mini, qwen3.6-35b-a3b, gemma-4-31b",106,106,317,2,3,0.074,0.409,0.818
6,Completeness,method,ours,Ours,interval_squared,3,3,"gpt-5.4-mini, qwen3.6-35b-a3b, gemma-4-31b",106,106,317,2,3,0.073,0.332,0.781
8,Relevance,method,llmbased,GWB,interval_squared,3,3,"gpt-5.4-mini, qwen3.6-35b-a3b, gemma-4-31b",106,106,318,3,3,0.077,0.131,0.415
9,Relevance,method,vectorsimilarity,VSB,interval_squared,3,3,"gpt-5.4-mini, qwen3.6-35b-a3b, gemma-4-31b",106,106,318,3,3,0.177,0.336,0.474
10,Relevance,method,grasp,GRASP,interval_squared,3,3,"gpt-5.4-mini, qwen3.6-35b-a3b, gemma-4-31b",106,106,316,2,3,0.087,0.441,0.803
11,Relevance,method,hipporag,HippoRAG,interval_squared,3,3,"gpt-5.4-mini, qwen3.6-35b-a3b, gemma-4-31b",106,106,317,2,3,0.111,0.302,0.631
